# Fashion Outfit Similarity Finder
**Pipeline:** Fashion200k → Fashionpedia detector → garment crop extraction → EfficientNetV2-S fine-tuning with triplet loss → FAISS indexing → retrieval demo → UMAP visualization

---
### Run order
- **Sections 1–6:** run once to build the index. Heavy computation, save outputs to disk.
- **Sections 7–8:** run any time for retrieval demo and visualization.

### Swapping to DeepFashion later
Change `DATASET_ROOT` in Section 1 to your DeepFashion path. Everything else is identical.

---
## Section 1 — Imports & Configuration
All paths and hyperparameters live here. Change values here only, not inside the code.

In [ ]:
import os
import json
import random
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm                          # model zoo — gives us EfficientNetV2-S cleanly
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import faiss
import umap
from ultralytics import YOLO
from datasets import load_dataset    # HuggingFace datasets library

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT         = Path(".")
DATASET_ROOT = ROOT / "fashion200k"   # downloaded dataset goes here
CROPS_DIR    = ROOT / "fashion_crops" # extracted garment crops
EMB_DIR      = ROOT / "fashion_emb"   # vectors + metadata
INDEX_DIR    = ROOT / "fashion_index" # FAISS index
CKPT_DIR     = ROOT / "fashion_ckpt"  # model checkpoints

for d in [DATASET_ROOT, CROPS_DIR, EMB_DIR, INDEX_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Detection config ──────────────────────────────────────────────────────────
# Fashionpedia model detects 27 garment categories instead of generic "person"
YOLO_MODEL_PATH = "valentinafeve/yolov8n-fashionpedia"  # auto-downloads on first use
YOLO_CONF       = 0.35   # lower than COCO — fashion detection is harder
MIN_CROP_SIZE   = 48     # slightly larger minimum — garment details need more pixels

# ── Embedding model config ────────────────────────────────────────────────────
EMBED_DIM = 128          # output embedding dimension
IMG_SIZE  = 224          # EfficientNetV2-S default input size

# ── Training config ───────────────────────────────────────────────────────────
TRIPLET_MARGIN = 0.4     # slightly higher than object retrieval — fashion similarity is finer-grained
BATCH_SIZE     = 48      # slightly lower than COCO notebook — EfficientNetV2-S uses more memory
NUM_EPOCHS     = 15      # more epochs — fashion similarity is harder to learn than general objects
LR             = 1e-4    # lower LR for backbone — EfficientNetV2-S weights are sensitive
LR_HEAD        = 8e-4    # projection head LR
WEIGHT_DECAY   = 1e-4

# ── Retrieval config ──────────────────────────────────────────────────────────
TOP_K          = 5

# ── num_workers ───────────────────────────────────────────────────────────────
# Set to 0 on Windows — multiprocessing in DataLoader breaks on Windows with Jupyter
# Change to 4 on Linux for faster data loading
NUM_WORKERS    = 0

print("\nConfiguration loaded.")

---
## Section 2 — Download Fashion200k

Fashion200k is available on HuggingFace and downloads automatically — no registration needed.

The dataset has:
- ~200,000 fashion product images
- Product labels (category + description) used to construct positive pairs
- 5 categories: dresses, jackets, pants, skirts, tops

**To switch to DeepFashion later:**
1. Register at mmlab.ie.cuhk.edu.hk and download In-Shop Clothes Retrieval
2. Set `DATASET_ROOT` to your DeepFashion folder
3. Adjust the manifest-building cell to read DeepFashion's annotation format
4. Everything else (detection, training, retrieval) stays identical

In [ ]:
# ── Download Fashion200k from HuggingFace ─────────────────────────────────────
# This downloads ~15GB on first run and caches it locally.
# HuggingFace caches to ~/.cache/huggingface by default.
# Subsequent runs load from cache instantly.

print("Loading Fashion200k from HuggingFace...")
print("First run downloads ~15GB — this will take a while.")

# Load train split — has the most images
# streaming=False downloads everything locally for faster access during training
fashion_dataset = load_dataset(
    "rajuptvs/fashion200k",
    split="train",
    trust_remote_code=True
)

print(f"Dataset loaded. Total samples: {len(fashion_dataset)}")
print(f"Columns: {fashion_dataset.column_names}")

# Preview a few entries
print("\nSample entries:")
for i in range(3):
    sample = fashion_dataset[i]
    print(f"  [{i}] label: {sample.get('label', sample.get('category', 'N/A'))} | "
          f"keys: {list(sample.keys())}")

In [ ]:
# ── Save images locally and build initial manifest ────────────────────────────
# HuggingFace serves images as PIL Image objects.
# We save them to disk so the rest of the pipeline works with file paths
# consistently — same as the COCO notebook.
#
# For speed during development, we cap at MAX_IMAGES.
# Remove the cap for full training runs.

MAX_IMAGES = 100  # use 5000 for dev, set to len(fashion_dataset) for full run

images_dir = DATASET_ROOT / "images"
images_dir.mkdir(exist_ok=True)

raw_manifest = []  # {image_path, label, category}

print(f"Saving {MAX_IMAGES} images to disk...")
for idx in tqdm(range(min(MAX_IMAGES, len(fashion_dataset)))):
    sample = fashion_dataset[idx]

    # Get image — HuggingFace returns PIL Image directly
    img = sample["image"]
    if img is None:
        continue

    # Get label — column name varies by dataset version
    label    = sample.get("label", sample.get("category", sample.get("name", f"item_{idx}")))
    category = sample.get("category", str(label).split()[0] if label else "unknown")

    # Save image
    img_path = images_dir / f"{idx:06d}.jpg"
    img.convert("RGB").save(img_path, quality=90)

    raw_manifest.append({
        "image_path": str(img_path),
        "label": str(label),
        "category": str(category)
    })

print(f"Saved {len(raw_manifest)} images.")

# Show category distribution
cat_counts = defaultdict(int)
for m in raw_manifest:
    cat_counts[m["category"]] += 1
print("\nTop categories:")
for cat, count in sorted(cat_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {cat:25s}: {count}")

---
## Section 3 — Garment Detection & Crop Extraction

Unlike the COCO notebook where we detected general objects (dog, car, person),
here we use `valentinafeve/yolov8n-fashionpedia` — a YOLOv8 model fine-tuned on
the Fashionpedia dataset.

It detects 27 specific apparel categories:
shirt, trouser, dress, coat, shoe, bag, hat, skirt, belt, glasses, etc.

This gives us garment-level crops instead of person-level crops.
A photo of a person wearing 3 items produces 3 separate crops — one per garment.
Each crop then gets embedded independently.

In [ ]:
# ── Load Fashionpedia detector ────────────────────────────────────────────────
# First run downloads the model weights from HuggingFace automatically
print("Loading Fashionpedia detector...")
fashion_detector = YOLO(YOLO_MODEL_PATH)

# Print detected categories so you know what the model sees
print(f"\nDetectable garment categories ({len(fashion_detector.names)}):")
for idx, name in fashion_detector.names.items():
    print(f"  {idx:2d}: {name}")

In [ ]:
# ── Run detection and extract crops ──────────────────────────────────────────
# For each image:
#   1. Run Fashionpedia detector — get bounding boxes for each garment
#   2. Crop each bounding box out of the original image
#   3. Save crop as PNG
#   4. Record in manifest: crop path, garment type, product label, source image
#
# Why crop garments separately:
#   If we embedded whole images, the embedding would pick up on background,
#   the person's face, and other irrelevant features.
#   Cropping to just the garment forces the embedding to focus on
#   color, texture, pattern and cut — the actual fashion features.

crop_manifest = []  # {crop_path, garment_type, product_label, category, source_image}

print(f"Running Fashionpedia detection on {len(raw_manifest)} images...")
print("This is the slowest step — runs once only.")

for item in tqdm(raw_manifest):
    img_path = item["image_path"]

    try:
        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        # Run detector
        results = fashion_detector.predict(
            source=img_path,
            conf=YOLO_CONF,
            save=False,
            verbose=False
        )

        result = results[0]  # one result per image

        if len(result.boxes) == 0:
            # No garments detected — fall back to whole image crop
            # This happens for flat-lay product photos with no person
            crop_path = CROPS_DIR / f"full_{Path(img_path).stem}.png"
            img.save(crop_path)
            crop_manifest.append({
                "crop_path": str(crop_path),
                "garment_type": item["category"],
                "product_label": item["label"],
                "category": item["category"],
                "source_image": img_path
            })
            continue

        boxes      = result.boxes.xyxy.cpu().numpy()
        class_ids  = result.boxes.cls.cpu().numpy().astype(int)
        garment_names = [result.names[c] for c in class_ids]

        for box_idx, (box, garment_name) in enumerate(zip(boxes, garment_names)):
            x1, y1, x2, y2 = box
            x1, y1 = max(0, int(x1)), max(0, int(y1))
            x2, y2 = min(w, int(x2)), min(h, int(y2))

            # Skip crops that are too small
            if (x2 - x1) < MIN_CROP_SIZE or (y2 - y1) < MIN_CROP_SIZE:
                continue

            crop = img.crop((x1, y1, x2, y2))
            stem = Path(img_path).stem
            crop_filename = f"{garment_name}_{stem}_{box_idx:02d}.png"
            crop_path = CROPS_DIR / crop_filename
            crop.save(crop_path)

            crop_manifest.append({
                "crop_path": str(crop_path),
                "garment_type": garment_name,    # what fashionpedia detected
                "product_label": item["label"],  # original product label from dataset
                "category": item["category"],    # top-level category (dress, top, etc.)
                "source_image": img_path
            })

    except Exception as e:
        print(f"Skipped {img_path}: {e}")
        continue

# Save manifest
with open(EMB_DIR / "crop_manifest.json", "w") as f:
    json.dump(crop_manifest, f, indent=2)

print(f"\nExtracted {len(crop_manifest)} garment crops.")

# Show garment type distribution
garment_counts = defaultdict(int)
for m in crop_manifest:
    garment_counts[m["garment_type"]] += 1
print("\nGarment types found:")
for gtype, count in sorted(garment_counts.items(), key=lambda x: -x[1]):
    print(f"  {gtype:25s}: {count}")

---
## Section 4 — Model Architecture (EfficientNetV2-S)

### Why EfficientNetV2-S instead of ResNet50

ResNet50 was designed for general object recognition where **shape** is the dominant feature.
Fashion similarity depends heavily on **texture, color, and pattern** — a striped linen blazer
should be close to other striped linen blazers, not just other blazers in general.

EfficientNetV2-S uses compound scaling — it balances depth, width and resolution simultaneously.
Its channel-level features are richer, capturing subtle texture and color distributions better.
It also trains faster and generalizes better on smaller datasets like Fashion200k.

### Architecture
```
Input (3 × 224 × 224)
→ EfficientNetV2-S backbone (frozen blocks 0-4, unfrozen blocks 5-6)
→ Global average pool → 1280-dim vector
→ Projection head: Linear(1280→512) → BN → ReLU → Linear(512→128)
→ L2 normalization
→ 128-dim unit vector (the embedding)
```

In [ ]:
class FashionEmbeddingModel(nn.Module):
    """
    EfficientNetV2-S backbone + projection head for fashion metric learning.

    EfficientNetV2-S has 7 fused-MBConv/MBConv stages.
    We freeze stages 0-4 (basic vision features) and unfreeze stages 5-6
    (high-level semantic features most relevant to fashion similarity).
    """
    def __init__(self, embed_dim=128):
        super().__init__()

        # Load EfficientNetV2-S pretrained on ImageNet-21k then fine-tuned on ImageNet-1k
        # in21k pretraining gives better features than 1k-only for fine-grained tasks
        self.backbone = timm.create_model(
            "tf_efficientnetv2_s.in21k_ft_in1k",
            pretrained=True,
            num_classes=0,        # remove classification head, output raw features
            global_pool="avg"     # global average pool after last conv block → 1280-dim
        )

        # Freeze all backbone parameters first
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Unfreeze the last two blocks (blocks[5] and blocks[6])
        # These capture high-level semantic and texture features
        for block in list(self.backbone.blocks)[-2:]:
            for param in block.parameters():
                param.requires_grad = True

        # Also unfreeze the final conv + bn layer
        for param in self.backbone.conv_head.parameters():
            param.requires_grad = True
        for param in self.backbone.bn2.parameters():
            param.requires_grad = True

        # Get backbone output dimension (1280 for EfficientNetV2-S)
        backbone_dim = self.backbone.num_features

        # Projection head — trained entirely from scratch
        # Compresses 1280 → 128 while learning fashion-specific similarity
        self.projection = nn.Sequential(
            nn.Linear(backbone_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),          # dropout helps prevent overfitting on fashion200k
            nn.Linear(512, embed_dim)
        )

    def forward(self, x):
        features   = self.backbone(x)               # (batch, 1280)
        embeddings = self.projection(features)       # (batch, 128)
        embeddings = F.normalize(embeddings, p=2, dim=1)  # unit sphere
        return embeddings


model = FashionEmbeddingModel(embed_dim=EMBED_DIM).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}  (last 2 blocks + conv_head + projection)")
print(f"Frozen parameters:    {frozen_params:,}  (blocks 0-4)")

---
## Section 5 — Triplet Dataset & Training

### Positive pair construction for Fashion200k

Fashion200k pairs by **product label** — two crops with the same product label
(e.g. both tagged "blue striped linen blazer") are positives.
Two crops with different labels are negatives.

This is slightly less precise than DeepFashion where positives are literally
the same physical item photographed twice. But it works well in practice —
same product label means same garment type, color and pattern.

### Hard negative mining

After a few epochs of random negatives, the model stops learning because
easy negatives (a dress vs a shoe) produce zero loss. We switch to
hard negatives — garments of different labels but **same category**
(e.g. two different blue blazers). These are genuinely confusing for the model
and force it to learn fine-grained fashion distinctions.

In [ ]:
# ── Image transforms ──────────────────────────────────────────────────────────
# Fashion-specific augmentations:
# - RandomHorizontalFlip: garments look the same mirrored
# - ColorJitter: lighting varies between product photos
# - RandomRotation: slight rotation for robustness
# - RandomResizedCrop: simulates different zoom levels on same garment
#
# No vertical flip — garments have a clear up/down orientation

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),  # slightly larger before crop
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

inference_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transforms defined.")

In [ ]:
# ── Fashion Triplet Dataset ───────────────────────────────────────────────────
class FashionTripletDataset(Dataset):
    """
    Builds triplets from garment crops using product labels.

    Positive pair:  same product_label  (same item, possibly different crop angle)
    Negative pair:  different product_label

    Hard negative mode: negative is chosen from the SAME category as the anchor
    (e.g. anchor=blue blazer, hard negative=green blazer, not a shoe)
    This forces the model to learn fine-grained differences within a category.
    """
    def __init__(self, manifest, transform=None, hard_negative=False):
        self.manifest      = manifest
        self.transform     = transform
        self.hard_negative = hard_negative

        # Group by product label for positive sampling
        self.label_to_indices = defaultdict(list)
        for idx, item in enumerate(manifest):
            self.label_to_indices[item["product_label"]].append(idx)

        # Group by category for hard negative sampling
        self.category_to_labels = defaultdict(set)
        for item in manifest:
            self.category_to_labels[item["category"]].add(item["product_label"])

        # Only keep labels with at least 2 crops
        self.valid_labels = [
            label for label, indices in self.label_to_indices.items()
            if len(indices) >= 2
        ]

        # All labels for random negative fallback
        self.all_labels = list(self.label_to_indices.keys())

        print(f"Total crops: {len(manifest)}")
        print(f"Unique product labels: {len(self.all_labels)}")
        print(f"Labels with ≥2 crops (usable as anchors): {len(self.valid_labels)}")
        print(f"Hard negative mode: {hard_negative}")

    def set_hard_negative(self, val: bool):
        """Switch hard negative mining on/off during training."""
        self.hard_negative = val
        print(f"Hard negative mining: {'ON' if val else 'OFF'}")

    def _load_image(self, idx):
        path = self.manifest[idx]["crop_path"]
        img  = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, idx):
        anchor_item  = self.manifest[idx]
        anchor_label = anchor_item["product_label"]

        # If anchor label doesn't have a positive, pick a random valid anchor instead
        if anchor_label not in self.label_to_indices or \
           len(self.label_to_indices[anchor_label]) < 2:
            anchor_label = random.choice(self.valid_labels)
            idx = random.choice(self.label_to_indices[anchor_label])
            anchor_item = self.manifest[idx]

        # Positive — same label, different index
        pos_candidates = [i for i in self.label_to_indices[anchor_label] if i != idx]
        pos_idx = random.choice(pos_candidates)

        # Negative
        if self.hard_negative:
            # Hard: different label but same category
            anchor_category = anchor_item["category"]
            same_cat_labels = [
                l for l in self.category_to_labels[anchor_category]
                if l != anchor_label
            ]
            if same_cat_labels:
                neg_label = random.choice(same_cat_labels)
            else:
                # Fallback to random if no other label in same category
                neg_label = random.choice([l for l in self.all_labels if l != anchor_label])
        else:
            # Easy: random label from any category
            neg_label = random.choice([l for l in self.all_labels if l != anchor_label])

        neg_idx = random.choice(self.label_to_indices[neg_label])

        return (
            self._load_image(idx),
            self._load_image(pos_idx),
            self._load_image(neg_idx),
            anchor_label
        )


# Load manifest and build dataset
with open(EMB_DIR / "crop_manifest.json") as f:
    crop_manifest = json.load(f)

# Start with easy (random) negatives — switch to hard after epoch 5
dataset = FashionTripletDataset(
    crop_manifest,
    transform=train_transform,
    hard_negative=False
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

In [ ]:
# ── Triplet Loss ──────────────────────────────────────────────────────────────
class TripletLoss(nn.Module):
    """
    L(a, p, n) = max(0, d(a,p) - d(a,n) + margin)

    d() = Euclidean distance on L2-normalized vectors
    (equivalent to cosine distance since vectors are unit-normalized)

    The loss is zero when the negative is already further from the anchor
    than the positive by at least the margin.
    Only violated triplets (loss > 0) produce gradients and cause learning.
    """
    def __init__(self, margin=0.4):
        super().__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        dist_pos   = torch.norm(anchor - positive, p=2, dim=1)  # (batch,)
        dist_neg   = torch.norm(anchor - negative, p=2, dim=1)  # (batch,)
        losses     = F.relu(dist_pos - dist_neg + self.margin)  # (batch,)
        active_frac = (losses > 0).float().mean().item()
        return losses.mean(), active_frac


criterion = TripletLoss(margin=TRIPLET_MARGIN)

# Two parameter groups — different learning rates for backbone vs projection head
# Backbone gets lower LR: its weights are already good, we nudge them gently
# Projection head gets higher LR: starts from random, needs bigger steps
optimizer = torch.optim.AdamW([
    {
        "params": [
            p for block in list(model.backbone.blocks)[-2:]
            for p in block.parameters()
        ] + list(model.backbone.conv_head.parameters())
          + list(model.backbone.bn2.parameters()),
        "lr": LR
    },
    {
        "params": model.projection.parameters(),
        "lr": LR_HEAD
    }
], weight_decay=WEIGHT_DECAY)

# Cosine annealing: LR starts high, smoothly decays to near-zero over NUM_EPOCHS
# Prevents oscillating around a good solution at the end of training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)

print("Loss, optimizer and scheduler ready.")

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────────
# Each epoch:
#   1. Forward: embed anchor, positive, negative
#   2. Compute triplet loss
#   3. Backward: compute gradients for unfrozen parameters
#   4. Optimizer step: update weights
#   5. Scheduler step: decay LR
#
# Hard negative switch:
#   Epochs 1-5:  random negatives   — model learns basic category separation
#   Epochs 6-15: hard negatives     — model learns fine-grained fashion similarity
#
# This two-phase approach prevents the model from getting stuck on hard negatives
# too early when it hasn't learned the basics yet.

HARD_NEGATIVE_EPOCH = 5   # switch to hard negatives after this epoch

history   = {"loss": [], "active_fraction": []}
best_loss = float("inf")

print(f"Training for {NUM_EPOCHS} epochs.")
print(f"Epochs 1-{HARD_NEGATIVE_EPOCH}: random negatives")
print(f"Epochs {HARD_NEGATIVE_EPOCH+1}-{NUM_EPOCHS}: hard negatives\n")

for epoch in range(NUM_EPOCHS):

    # Switch to hard negatives halfway through
    if epoch == HARD_NEGATIVE_EPOCH:
        dataset.set_hard_negative(True)

    model.train()
    epoch_loss   = 0.0
    epoch_active = 0.0
    num_batches  = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)
    for anchors, positives, negatives, _ in pbar:
        anchors   = anchors.to(DEVICE)
        positives = positives.to(DEVICE)
        negatives = negatives.to(DEVICE)

        optimizer.zero_grad()

        # Forward pass — embed all three images through shared model
        emb_a = model(anchors)
        emb_p = model(positives)
        emb_n = model(negatives)

        loss, active_frac = criterion(emb_a, emb_p, emb_n)

        # Backward pass
        loss.backward()

        # Gradient clipping — prevents large weight updates that destabilize training
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        epoch_loss   += loss.item()
        epoch_active += active_frac
        num_batches  += 1

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "active": f"{active_frac:.1%}"
        })

    avg_loss   = epoch_loss   / num_batches
    avg_active = epoch_active / num_batches
    history["loss"].append(avg_loss)
    history["active_fraction"].append(avg_active)

    scheduler.step()

    mode_str = "hard neg" if epoch >= HARD_NEGATIVE_EPOCH else "rand neg"
    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} [{mode_str}] | "
          f"Loss: {avg_loss:.4f} | "
          f"Active: {avg_active:.1%} | "
          f"LR: {scheduler.get_last_lr()[0]:.2e}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), CKPT_DIR / "best_model.pth")
        print(f"  ✓ Best model saved (loss: {best_loss:.4f})")

print(f"\nTraining complete. Best loss: {best_loss:.4f}")

# Plot training curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["loss"], marker="o")
ax1.axvline(x=HARD_NEGATIVE_EPOCH, color="red", linestyle="--", label="hard neg start")
ax1.set_title("Triplet Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True)

ax2.plot(history["active_fraction"], marker="o", color="orange")
ax2.axvline(x=HARD_NEGATIVE_EPOCH, color="red", linestyle="--", label="hard neg start")
ax2.set_title("Active Triplet Fraction")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Fraction")
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(EMB_DIR / "training_curve.png", dpi=150)
plt.show()

---
## Section 6 — Embed All Crops & Build FAISS Index

Run once after training. Embeds every garment crop and indexes the vectors.
After this, retrieval is instant.

In [ ]:
# ── Load best model ───────────────────────────────────────────────────────────
model.load_state_dict(torch.load(CKPT_DIR / "best_model.pth", map_location=DEVICE))
model.eval()
print("Best model loaded.")

# ── Embed all crops ───────────────────────────────────────────────────────────
all_vectors  = []
all_metadata = []

print(f"Embedding {len(crop_manifest)} garment crops...")
with torch.no_grad():
    for item in tqdm(crop_manifest):
        try:
            img    = Image.open(item["crop_path"]).convert("RGB")
            tensor = inference_transform(img).unsqueeze(0).to(DEVICE)
            emb    = model(tensor).squeeze(0).cpu().numpy()
            all_vectors.append(emb)
            all_metadata.append(item)
        except Exception as e:
            print(f"Skipped {item['crop_path']}: {e}")

vectors_matrix = np.vstack(all_vectors).astype("float32")
print(f"Embedding matrix: {vectors_matrix.shape}")

np.save(EMB_DIR / "vectors.npy", vectors_matrix)
with open(EMB_DIR / "metadata.json", "w") as f:
    json.dump(all_metadata, f, indent=2)
print("Saved vectors and metadata.")

In [ ]:
# ── Build FAISS Index ─────────────────────────────────────────────────────────
# IndexFlatIP = exact inner product search
# Since vectors are L2-normalized, inner product = cosine similarity
# Higher score = more similar garments
#
# For very large datasets (millions of crops) switch to IndexIVFFlat
# for approximate but much faster search.

dimension = vectors_matrix.shape[1]  # 128
index     = faiss.IndexFlatIP(dimension)

# Move to GPU
try:
    res   = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, index)
    print("FAISS running on GPU.")
except Exception:
    print("FAISS GPU unavailable, using CPU.")

index.add(vectors_matrix)
print(f"Indexed {index.ntotal} garment vectors.")

# Save to disk
try:
    cpu_index = faiss.index_gpu_to_cpu(index)
except Exception:
    cpu_index = index

faiss.write_index(cpu_index, str(INDEX_DIR / "fashion.index"))
print(f"Index saved to {INDEX_DIR / 'fashion.index'}")

---
## Section 7 — Retrieval Demo

Give it one garment crop → returns the 5 most visually similar garments from the database.

Change `query_crop_path` to any image from your `fashion_crops/` folder.
You can also point it at any new fashion photo — the detector will crop garments from it automatically.

In [ ]:
# ── Load everything ───────────────────────────────────────────────────────────
cpu_index = faiss.read_index(str(INDEX_DIR / "fashion.index"))
try:
    res   = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
except Exception:
    index = cpu_index

with open(EMB_DIR / "metadata.json") as f:
    all_metadata = json.load(f)

model.load_state_dict(torch.load(CKPT_DIR / "best_model.pth", map_location=DEVICE))
model.eval()
print("Ready for retrieval.")


# ── Helper functions ──────────────────────────────────────────────────────────
def embed_image(img_path):
    """Embed a single image and return its 128-dim vector."""
    img    = Image.open(img_path).convert("RGB")
    tensor = inference_transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        emb = model(tensor).squeeze(0).cpu().numpy()
    return emb.astype("float32")


def retrieve(query_path, top_k=TOP_K):
    """Find the top_k most similar garments to the query image."""
    query_vec = embed_image(query_path).reshape(1, -1)
    scores, indices = index.search(query_vec, top_k + 1)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        meta = all_metadata[idx]
        if meta["crop_path"] == str(query_path):
            continue  # skip the query itself if it's in the index
        results.append((meta, float(score)))
        if len(results) == top_k:
            break
    return results


def detect_and_retrieve(image_path, top_k=TOP_K):
    """
    Full pipeline on a new photo:
    1. Run Fashionpedia detector on image
    2. For each detected garment, retrieve top_k similar items
    3. Display results per garment
    """
    img = Image.open(image_path).convert("RGB")
    w, h = img.size

    results_all = fashion_detector.predict(
        source=str(image_path), conf=YOLO_CONF, save=False, verbose=False
    )
    result = results_all[0]

    if len(result.boxes) == 0:
        print("No garments detected. Treating whole image as query.")
        results = retrieve(image_path, top_k)
        _display_results(image_path, "whole image", results)
        return

    boxes      = result.boxes.xyxy.cpu().numpy()
    class_ids  = result.boxes.cls.cpu().numpy().astype(int)
    garments   = [result.names[c] for c in class_ids]

    for box, garment_name in zip(boxes, garments):
        x1, y1, x2, y2 = [max(0, int(v)) for v in box]
        x2, y2 = min(w, x2), min(h, y2)
        if (x2-x1) < MIN_CROP_SIZE or (y2-y1) < MIN_CROP_SIZE:
            continue

        crop_path = CROPS_DIR / f"query_{garment_name}_temp.png"
        img.crop((x1, y1, x2, y2)).save(crop_path)

        results = retrieve(crop_path, top_k)
        _display_results(crop_path, garment_name, results)


def _display_results(query_path, label, results):
    """Display query + top-K results as a matplotlib grid."""
    n = len(results) + 1
    fig, axes = plt.subplots(1, n, figsize=(3*n, 4))
    if n == 1:
        axes = [axes]

    axes[0].imshow(Image.open(query_path))
    axes[0].set_title(f"QUERY\n{label}", fontsize=9, color="steelblue", fontweight="bold")
    axes[0].axis("off")

    for i, (meta, score) in enumerate(results):
        axes[i+1].imshow(Image.open(meta["crop_path"]))
        axes[i+1].set_title(
            f"#{i+1} {meta['garment_type']}\n{score:.3f}",
            fontsize=8
        )
        axes[i+1].axis("off")

    plt.suptitle(f"Query: {label} → Top {len(results)} similar garments", fontsize=11)
    plt.tight_layout()
    plt.savefig(EMB_DIR / f"retrieval_{label}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nResults for: {label}")
    for i, (meta, score) in enumerate(results):
        print(f"  #{i+1} | {meta['garment_type']:20s} | sim: {score:.4f} | {Path(meta['crop_path']).name}")


# ── Run a retrieval ───────────────────────────────────────────────────────────
# Option A: query with a single crop from your crops folder
query_crop_path = sorted(CROPS_DIR.glob("*.png"))[0]  # first crop alphabetically
print(f"Querying with: {query_crop_path.name}")
results = retrieve(query_crop_path)
_display_results(query_crop_path, query_crop_path.stem[:20], results)

# Option B: query with a new photo — detects garments automatically
# detect_and_retrieve("path/to/your/outfit/photo.jpg")

---
## Section 8 — UMAP Visualization & Embedding Analysis

In [ ]:
# ── Load vectors ──────────────────────────────────────────────────────────────
vectors_matrix = np.load(EMB_DIR / "vectors.npy")
with open(EMB_DIR / "metadata.json") as f:
    all_metadata = json.load(f)

garment_types = [m["garment_type"] for m in all_metadata]
categories    = [m["category"]     for m in all_metadata]

print(f"Loaded {vectors_matrix.shape[0]} vectors, dim {vectors_matrix.shape[1]}")


# ── Run UMAP ──────────────────────────────────────────────────────────────────
# Projects 128-dim vectors to 2D while preserving local structure.
# Similar garments stay close together in 2D.
# n_neighbors=15: balances local vs global structure
# min_dist=0.05: tighter clusters (good for fashion where intra-class variance is low)
# metric=cosine: correct for L2-normalized vectors

print("Running UMAP...")
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.05,
    metric="cosine",
    random_state=42,
    verbose=True
)
embedding_2d = reducer.fit_transform(vectors_matrix)
print(f"UMAP complete: {embedding_2d.shape}")


# ── Plot by garment type ──────────────────────────────────────────────────────
unique_types = sorted(set(garment_types))
cmap = plt.cm.get_cmap("tab20", len(unique_types))
type_to_color = {t: cmap(i) for i, t in enumerate(unique_types)}

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Left plot: colored by garment type (shirt, trouser, dress etc.)
for gtype in unique_types:
    mask = np.array([t == gtype for t in garment_types])
    axes[0].scatter(
        embedding_2d[mask, 0],
        embedding_2d[mask, 1],
        c=[type_to_color[gtype]],
        label=gtype,
        s=15,
        alpha=0.7
    )
axes[0].set_title("Garment embeddings — by type", fontsize=13)
axes[0].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7, markerscale=2)
axes[0].axis("off")

# Right plot: colored by top-level category (dress, top, pants etc.)
unique_cats = sorted(set(categories))
cmap2 = plt.cm.get_cmap("Set1", len(unique_cats))
cat_to_color = {c: cmap2(i) for i, c in enumerate(unique_cats)}

for cat in unique_cats:
    mask = np.array([c == cat for c in categories])
    axes[1].scatter(
        embedding_2d[mask, 0],
        embedding_2d[mask, 1],
        c=[cat_to_color[cat]],
        label=cat,
        s=15,
        alpha=0.7
    )
axes[1].set_title("Garment embeddings — by category", fontsize=13)
axes[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9, markerscale=2)
axes[1].axis("off")

plt.suptitle("Fashion Embedding Space (UMAP projection)", fontsize=15)
plt.tight_layout()
plt.savefig(EMB_DIR / "umap_fashion.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"UMAP saved.")

In [ ]:
# ── Embedding values visualization ────────────────────────────────────────────
# Compare raw embedding vectors of:
#   A) Two similar garments (same type) — should have high cosine similarity
#   B) Two different garments (different type) — should have low cosine similarity
#
# This is how you show the actual numbers of the embedding.

# Find two crops of the same garment type
type_to_idxs = defaultdict(list)
for i, m in enumerate(all_metadata):
    type_to_idxs[m["garment_type"]].append(i)

# Pick a garment type that has at least 2 crops
sample_type = next(t for t, idxs in type_to_idxs.items() if len(idxs) >= 2)
idx_same_a  = type_to_idxs[sample_type][0]
idx_same_b  = type_to_idxs[sample_type][1]

# Pick a different garment type for contrast
diff_type  = next(t for t in type_to_idxs if t != sample_type)
idx_diff   = type_to_idxs[diff_type][0]

vec_a = vectors_matrix[idx_same_a]
vec_b = vectors_matrix[idx_same_b]
vec_c = vectors_matrix[idx_diff]

sim_same = float(np.dot(vec_a, vec_b))   # same type — should be high
sim_diff = float(np.dot(vec_a, vec_c))   # different type — should be low

print(f"Garment A: {all_metadata[idx_same_a]['garment_type']}")
print(f"Garment B: {all_metadata[idx_same_b]['garment_type']} (same type)")
print(f"Garment C: {all_metadata[idx_diff]['garment_type']} (different type)")
print(f"\nCosine similarity A↔B (same):      {sim_same:.4f}")
print(f"Cosine similarity A↔C (different):  {sim_diff:.4f}")
print(f"\nFirst 10 values of embedding A: {vec_a[:10].round(4)}")

# Bar charts
fig, axes = plt.subplots(3, 1, figsize=(14, 8))

axes[0].bar(range(EMBED_DIM), vec_a, color="steelblue", alpha=0.8)
axes[0].set_title(f"Embedding A — {all_metadata[idx_same_a]['garment_type']}")

axes[1].bar(range(EMBED_DIM), vec_b, color="seagreen", alpha=0.8)
axes[1].set_title(
    f"Embedding B — {all_metadata[idx_same_b]['garment_type']} "
    f"(same type) | cosine sim with A: {sim_same:.4f}"
)

axes[2].bar(range(EMBED_DIM), vec_c, color="coral", alpha=0.8)
axes[2].set_title(
    f"Embedding C — {all_metadata[idx_diff]['garment_type']} "
    f"(different type) | cosine sim with A: {sim_diff:.4f}"
)

for ax in axes:
    ax.set_xlabel("Dimension")
    ax.set_ylabel("Value")

plt.tight_layout()
plt.savefig(EMB_DIR / "embedding_values.png", dpi=150)
plt.show()